In [3]:
# magics: ensures that any changes to the modules loaded below will be re-loaded automatically
%load_ext autoreload
%autoreload 2

# load general packages
import os
#os.chdir('/Users/jacobaspnissen/Desktop/Økonomi/Dynamic programming/Termpaper/termpaper_dynprog')
os.chdir('/Users/albertolsen/Documents/Stud.Polit./9. semester/Dynamic Programming/Exam/termpaper_dynprog')

import numpy as np
import time
import copy
import pandas as pd
import pyreadr
import pickle
import xarray as xr  # Ensure xarray is installed
import statsmodels.api as sm



import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
plt.style.use('seaborn-v0_8-whitegrid')

# load modules related EGM
#import tools 
#from model_exante import model_bufferstock
#import estimate_exante as estimate

# load modules related to NFXP
from model_retirement import retirement
from Solve_NFXP import solve_NFXP
import estimate_NFXP as estimate

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ModuleNotFoundError: No module named 'pyreadr'

# Import Data 

In [3]:
# Clean data
cleaned_lines = []
with open("KPS Data/sparadata2.txt", "r") as file:
    for line in file:
        # Replace multiple spaces with a single tab
        cleaned_line = " ".join(line.split())  # Normalize spaces
        cleaned_lines.append(cleaned_line)

# Save the cleaned file
with open("KPS Data/cleaned_sparadata.txt", "w") as cleaned_file:
    cleaned_file.write("\n".join(cleaned_lines))

In [4]:
datapath = "KPS Data/cleaned_sparadata.txt"
sparadata = np.genfromtxt(open(datapath, "rb"), delimiter=" ", skip_header=0, dtype=None, encoding=None)


- index   is an identifier for the observation (each individual and year)
- id2    	is an identifier for the individual
- sex    	1=man 2=woman
- year   	of observation
- age    	of the individual for the given year
- married	marital status
- retire	yes=1, no=0
- income	1/1000 SEK. This number is rounded (no decimal digits) due to
	confidentiality reasons
- atp	is the "average pension points" used in the paper

In [10]:
data = pd.read_csv(datapath, sep=" ", header=0, encoding='utf-8')
data = data.drop(columns="index")
# create data['atp_1'] as the atp for the same individual in the next period using id to check if individual is the same and year to check if it is the next period
data['atp_1'] = data.groupby('id')['atp'].shift(-1)  
data['atp_1'] = data['atp_1'].fillna(0)  # Fill NaN values with 0 for atp_1
data['log_atp'] = np.log(data['atp'])
data['age_squared'] = data['age'] ** 2
data

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,id,sex,year,age,married,ret,income,atp,atp_1,log_atp,age_squared
0,3,1,83,52,1,0.0,179,4.636000,4.636000,1.533852,2704
1,3,1,84,53,1,0.0,188,4.636000,4.636000,1.533852,2809
2,3,1,85,54,1,0.0,186,4.636000,4.636000,1.533852,2916
3,3,1,86,55,1,0.0,189,4.636000,4.640667,1.533852,3025
4,3,1,87,56,1,0.0,196,4.640667,4.658000,1.534858,3136
...,...,...,...,...,...,...,...,...,...,...,...
51366,44094,2,92,52,1,0.0,154,2.803333,2.875333,1.030809,2704
51367,44094,2,93,53,1,0.0,154,2.875333,2.944667,1.056169,2809
51368,44094,2,94,54,0,0.0,154,2.944667,3.009333,1.079996,2916
51369,44094,2,95,55,0,0.0,154,3.009333,3.064667,1.101719,3025


In [21]:
# State variables: age, wage, average atp points, retirement age and marital status
# Regularize to avoid log(0)
epsilon = 1e-4
data2 = data[data['atp'] > 0]  # Remove 0s to avoid -inf
data2 = data2[data2['atp_1'] > 0]  # Remove 0s in atp_1 to avoid -inf


# Construct regression variables
X = sm.add_constant(data2[['log_atp', 'age', 'age_squared']])

y = np.log(data2['atp_1'])

# Estimate OLS
model = sm.OLS(y, X).fit()
print(model.summary())

# Extract coefficients and residual variance
gamma = model.params
sigma2 = model.mse_resid

                            OLS Regression Results                            
Dep. Variable:                  atp_1   R-squared:                       0.997
Model:                            OLS   Adj. R-squared:                  0.997
Method:                 Least Squares   F-statistic:                 4.701e+06
Date:                Fri, 01 Aug 2025   Prob (F-statistic):               0.00
Time:                        15:53:48   Log-Likelihood:             1.0493e+05
No. Observations:               46733   AIC:                        -2.099e+05
Df Residuals:                   46729   BIC:                        -2.098e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -0.0428      0.024     -1.791      

In [26]:
apt_next = np.exp(X @ gamma + sigma2/2)  # Predict next period ATP points

In [27]:
apt_next

0        4.684475
1        4.682590
2        4.680317
3        4.677656
4        4.679187
           ...   
51365    2.802719
51366    2.870957
51367    2.941517
51368    3.009068
51369    3.071618
Length: 46733, dtype: float64

#### Consider the retirement model given by:

$$
V(x_t,\varepsilon_t , \theta) = \max_d\in \{D(x_t)\} \big\{ u(x,d) + \varepsilon_d + \beta
\underbrace{\int_{X} \int_{\Omega} V(x',\varepsilon') \pi(x'|x,d) q(\varepsilon'|x') dx' d\varepsilon' }_{EV(x,d)} \big\}
$$

$$
V(x_t,\varepsilon_t , \theta) = \max_{d\in \{0,1\}} \big\{ u(x,d) + \varepsilon_d + \beta
\underbrace{\int_{X} \int_{\Omega} V(x',\varepsilon') \pi(x'|x,d) q(\varepsilon'|x') dx' d\varepsilon' }_{EV(x,d)} \big\}
$$

Where $ \varepsilon $ is extreme value Type I distribued and utility is given by:

$$
u(x,d)=\left \{
\begin{array}{ll}
    -RC-c(0,\theta_1) & \text{if }d=\text{replace}=1 \\
    -c(x,\theta_1) & \text{if }d=\text{keep}=0
\end{array} \right.
$$

Here

- $ RC $ = replacement cost  
- $ c(x,\theta_1) $ = cost of maintenance with preference parameters $ \theta_1 $  




We get our transition probability vector, that is the transition probabilities stated in the setup, which is an array. Then we sum all probabilities and deduct from 1, leaving us with the probability for passing onto the last state, let's say we state probabilities of staying in state 0, moving to 1 or 2, but this does not sum to 1, then probability of moving to state 3 will be 1 - sum of all the stated probabilities.
    
- "Then we initialize the state transition matrix as a N x N matrix, with zeros.\n",

- "We loop over each row of the transition matrix:\n",
1. for iteration in range n, we check if the probability vector fits entirely in the row, if that is the case, we insert that at point i until the end of the p vector, imagine p vector is 4 x 1 and matrix is 12 x 12 (range 0 to 11), in the first iteration we then insert the matrix in row i (0), column i (0) and to column 3. in next iteration we interst p vector from row 1, column 1 to 4 and so forth.\n",
    "When the vector no longer fits, we insert in row (i) and colum (i) until the end, but the probabilities are summed \"backwards\". lets say we are in iteration 9, we then insert vector in row 9, first item in vector is inserted in column 9, second item in 10, and third and fourth are summed in column 11.\n",
    "\n",
    "Then we create transition probaiblity if engine is replaced as this places us back in the state 0, we create the N x N matrix, but with the probability vector for being in state 0.\n",
    "\n",
    "\n",

### Zurcher.setup\n",
    "When setting up the problem we go through multiply stepts\n",
    "a. we setup the parameters, first we define a gridspace by setting the n (number of gridpoints) and a max (in this case the max value of mileage) \n",
    "Then we setup the structural parameters:\n",
    "we set the transition probabilities (p), the replacement cost (RC), the cost (maintenance cost) parameter (c) and the discount factor (beta)\n",
    "\n",
    "b. in a. we set some placeholder values for each parameter, then we use the kwargs to update these values to the ones we specify for the baseline parameters\n",
    "\n",
    "c. we then call the function that creates the grid based on parameters\n",
    "\n",
    "\n",
    
    
    
### Zurcher.create_grid\n",
    "first, we create the mileage grid, that is a range that runs from 0 to n (the number of gridpoints we specified in the setup)\n",
    "\n",
    "second, we create a cost function, that is by multiplying the maintenance cost (c) with the mileage grid and some scaling parameter (in this case 0.001). So we have a function of maintenance cost that increases as mileage goes up.\n",
    "\n",
    "Third, with the mileage grid and the costfunction we find the state transition\n",
    "\n",
    "\n",

### Zurcher.state_transition\n",
    "We get our transition probability vector, that is the transition probabilities stated in the setup, which is an array. Then we sum all probabilities and deduct from 1, leaving us with the probability for passing onto the last state, let's say we state probabilities of staying in state 0, moving to 1 or 2, but this does not sum to 1, then probability of moving to state 3 will be 1 - sum of all the stated probabilities.\n",
    "\n",
    "Then we initialize the state transition matrix as a N x N matrix, with zeros.\n",
    "\n",
    "We loop over each row of the transition matrix:\n",
    "1. for iteration in range n, we check if the probability vector fits entirely in the row, if that is the case, we insert that at point i until the end of the p vector, imagine p vector is 4 x 1 and matrix is 12 x 12 (range 0 to 11), in the first iteration we then insert the matrix in row i (0), column i (0) and to column 3. in next iteration we interst p vector from row 1, column 1 to 4 and so forth.\n",
    "When the vector no longer fits, we insert in row (i) and colum (i) until the end, but the probabilities are summed \"backwards\". lets say we are in iteration 9, we then insert vector in row 9, first item in vector is inserted in column 9, second item in 10, and third and fourth are summed in column 11.\n",
    "\n",
    "Then we create transition probaiblity if engine is replaced as this places us back in the state 0, we create the N x N matrix, but with the probability vector for being in state 0.\n",
    "\n",
    "\n",
    
    
    
### Zurcher.bellman\n",
    "First we find the value of keeping, that is minus the maintenance cost + dot product of the expected value function from previous iteration (ev0) and the transition matrix of not replacing engine discounted at beta.\n",
    "Thus it can be seen as the immediate cost of keeping engine + the discounted expected future value of keeping the enginge. This is a n x 1 matrix.\n",
    "\n",
    "Second we find the value of replacing, that is the replacement cost and the maintenance cost in period 0 + the discounted expected future value of replacing the engine (beta times the dot product of the transition matrix of replacing and the expected value function from the previous iteration) This is a 1 x 1 matrix.\n",
    "\n",
    "After calculating value of keeping and value of replacing we evaluate and find MaxV:\n",
    "that is using np.max to find the maximum value between value of keeping and value of replacing\n",
    "\n",
    "We then compute the expected value over the two choices (keep or replace), that is logsum to handle expectation over unobserved states. It accounts for unobserved randomness in decision making process.\n",
    "That is the sum of the three: MaxV, the log sum of the exponentials of difference between value of keeping and maxV and difference between value of replacing and maxV (np.log(np.exp(value_keep - maxV) - np.exp(value_replace - maxV)))\n",
    "\n",
    "we then set the expected value function (ev1) equal to the logsum, that is updating expected value function after applying the bellman operator. \n",
    "\n",
    "In the bellman function we have the output parameter that controls what the bellman function returns.\n",
    "\n",
    "If output = 1 this returns the expected value function (ev1)\n",
    "\n",
    "If output = 2 the function returns expected value function (ev1) and choice probability of keeping enginge (pk)\n",
    "\n",
    "we compute choice probability of keeping engine\n",
    "        pk = 1/(1+np.exp(value_replace-value_keep))       \n",
    "\n",
    "and return ev1 and pk\n",
    "\n",
    "then compute derivative of the bellman operator by calling dbellman function: dev1 = self.dbellman(pk)\n",
    "\n",

### Zurcher.dbellman\n",
    "This function computes the derivstive of the bellman operator\n",
    "\n",
    "1 we set the derivative matrix, dev1, a N x N matrix of zeros, n being the number of states in the model. Each element of dev1[i, j] represents bellman operator for state i changes with respect to the expected value function for state j.\n",
    "\n",
    "2 We loop over the choices d==0 keep engine and d==1 replace engine.\n",
    "\n",
    "if d==0, the transition probability matrix, P, is set to P1 (prob. matrix for keeping engine) and choice probability is set to pk\n",
    "if d==1, the transition probability matrix, P, is set to p2 (prob. matrix for replacing) and choice probabilbity is set to 1 - pk.\n",
    "\n",
    "Then update derivative matrix, dev1 += self.beta * choice_prob.reshape(-1,1) * P\n",
    "- += : operator that adds value to existing variable\n",
    "- self.beta: discount factor which scales future value\n",
    "- choice_prob.reshape(-1,1): reshape choice probabilities from the loop into a N x 1 matrix\n",
    "- P : the transition probability matrix for the current choice\n",
    " \n"
   ]